# Simulación de temperatura de rollos para recocido en horno tipo campana.

Implementa la librería descrita en los archivos de Python.

## Imports

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from heattransfer.simulator import Simulator
from heattransfer.models.furnace import Furnace
from heattransfer.models.material import Material
from heattransfer.models.coil import Coil
from heattransfer.models.protection_gas import ProtectionGas
from heattransfer.analysis.analyzer import Analyzer

## Configuración

In [ ]:
# Steel SPCC-PSR
material = Material(
    density=7870.0,
    emissivity=0.08554,
    specific_heat=481.0,
    hardness=1133.86,
    roughness=3.2215e-6,
    conductivity=None,  # Taylor and Wang (2017), Eq. 26; T in kelvin
)

gas = ProtectionGas(
   conductivity=0.1805,
   viscosity=8.9e-6,
   density=0.0819
)

coil = Coil(
    material=material,
    inner_radius=0.605 / 2,
    outer_radius=1.837  / 2,
    strip_thickness=1.48e-3,
    height=0.636*2,
    point_separation=0.05,
    initial_temperature=(273.15 + 55),
    surface_slope=0.1,
    oxide_thickness=10e-6,
    oxide_conductivity=3.0,
    emulsion_correction=0.8
)

# Import sample temperature from file and clean it up
gas_temp = np.genfromtxt("./data/experimental/Thermocouples/1#.csv", delimiter=",")
gas_temp = gas_temp[~np.isnan(gas_temp).any(axis=1)]


furnace = Furnace(
    temperature=gas_temp,
    height=5.59, # (Zhang et. al)
    radius=1.03, # (Zhang et. al)
    coil_sep_height=0.10, # (Zhang et. al)
    coils=[coil],
    gas=gas,
    gas_flow=19.4, # (Zhang et. al)
    vortex_enhancement=1.0, # (Zhang et. al)
    radiation_enhancement=1.1,
)

simulator = Simulator(
    furnace = furnace, 
    time_step = 30.0, 
    simulation_time = furnace.duration
)

## Simulate
Activar simulator.run().

In [ ]:
results = simulator.run()

## Análisis

### Settings

Ajustes de la gráfica e importación de datos experimentales. 

In [ ]:
analyzer = Analyzer(results=results,
                    simulator=simulator, 
                    furnace=furnace)

r, z = 309e-3, 0.636

data = analyzer.get_point_data(r, z)

plot_height = 700


experimental_temp = np.genfromtxt(
    "./data/experimental/Thermocouples/11#.csv",
    delimiter=",",
    skip_header=1,
    ndmin=2,
)
experimental_temp = experimental_temp[np.isfinite(experimental_temp).all(axis=1)]


layout = {
    "title": {
        "text": f"Evolución de temperatura de punto en (z,r) = {z,r}",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.9,
        "yanchor": "top",
    },
    #"width": 1920,
    "height": plot_height,
    "font": {
        "family": "Arial",
        "size": 15,
        "color": "black",
    },
    "margin": {"t": int(plot_height * 0.25), "b": 100, "l": 100, "r": 100},
    "legend": {"x": 1.05, "xanchor": "left"},
    "xaxis": {"title": {"text": "Tiempo (h)"},
               "ticklabelstandoff": 15,
               "zeroline": False},
    "yaxis": {"ticklabelstandoff": 15},
    "template": "none"
}

fig = go.Figure(layout=layout)
fig.add_trace(go.Scatter(
    x=simulator.times / 3600.0,
    y=data,
    name="Punto objetivo"
))
fig.add_trace(go.Scatter(
    x=simulator.times / 3600.0,
    y=[furnace.temperature_at(time) for time in simulator.times],
    line = {"width": 2, 
            "dash": 'dash'},
    name="Gas de protección"
));


fig.add_trace(go.Scatter(
    x=experimental_temp[:, 0],
    y=experimental_temp[:, 1] + 273.15,
    line = {"width": 2, 
            "dash": "dot"},
    name="Termopar 11 (experimental)",
));


### Plot.

Evolución de la temperatura de punto de interés. 

In [ ]:
fig.show()